In [1]:
import os
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from etl.transformations.common import (
    create_spark,
    write_clickhouse,
    read_batches_since,
)
from etl.transformations.state import (
    get_watermark,
    set_watermark,
)

WATERMARK_PATH = "s3a://staging/_checkpoints/spark/fact_harvests/watermark.json"

SOURCE_PATH = "s3a://staging/raw/postgres/harvests/"

In [2]:
def main():

    spark = create_spark("fact_harvests")

    last_batch = get_watermark(
        spark,
        WATERMARK_PATH,
    )

    # read new batches
    harvests_df, new_batch = read_batches_since(
        spark,
        SOURCE_PATH,
        last_batch,
    )

    harvests_df.printSchema()
    harvests_df.show()
    print(new_batch)

    # transform

    # write ClickHouse

    # update watermark

In [3]:
if __name__ == "__main__":
    main()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/22 20:08:05 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/07/22 20:08:09 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


root
 |-- id: long (nullable = true)
 |-- farm_id: long (nullable = true)
 |-- crop_id: long (nullable = true)
 |-- quality_grade_id: long (nullable = true)
 |-- weight_kg: decimal(5,3) (nullable = true)
 |-- created_at: long (nullable = true)
 |-- updated_at: long (nullable = true)
 |-- harvest_date: date (nullable = true)

+------+-------+-------+----------------+---------+----------+----------+------------+
|    id|farm_id|crop_id|quality_grade_id|weight_kg|created_at|updated_at|harvest_date|
+------+-------+-------+----------------+---------+----------+----------+------------+
|600716|     44|      6|               3|    0.797|1768587540|1768587540|  2026-01-16|
|601446|     44|      7|               1|    1.539|1768568520|1768568520|  2026-01-16|
|602176|     44|      8|               1|    1.582|1768587780|1768587780|  2026-01-16|
|602906|     44|      9|               2|    1.439|1768526400|1768526400|  2026-01-16|
|603636|     44|     10|               1|    1.316|1768547520|17